# Temporal-view ablation — ROCCO committee revision (item 3)

Standalone companion mirroring the structure of `BETO_ablation.ipynb`.

**Question:** does the temporal view contribute anything to the multi-view graph?

As originally coded, `sim_time` used the **MinMax-scaled `year`** (values in `[0, 1]`),
so `|year_i - year_j| <= 1` held for *every* pair → an all-ones matrix. A constant
offset added to every entry cannot change the per-row k-NN ranking, so the temporal
view should have **no effect**. This notebook proves that and tests a corrected
real-date version, on the **same 8-fold `StratifiedKFold(random_state=42)`** split.

| Variant | Temporal view | Purpose |
|---|---|---|
| `current` | scaled-year `\|Δ\|<=1` (reproduces original ROCCO) | the degenerate, all-ones view |
| `2view` | none (text + domain only) | temporal view removed entirely |
| `corrected` | real month index `\|Δ_month\|<=1` | a temporal view that actually varies |
| `permuted` | real month index, dates **shuffled** | signal-destroying control |

Per fold, BETO is fine-tuned **once** (frozen lower-6, 3 epochs — ROCCO's exact text
config) and only the **graph** changes across variants, so any difference is
attributable to the graph alone (identical text embeddings + identical GAT init).

> **Pipeline note:** this notebook reproduces the **presented ROCCO pipeline** exactly
> — augmentation applied to the whole dataset *before* the split, `corpus_enc` kept in
> the metadata, same `random_state=42` folds and `torch.manual_seed(1000+fold)` GAT
> init — so the `current` variant matches the numbers already defended. The temporal
> finding (the scaled-year view is inert) is a property of the graph construction and
> holds regardless of these choices.

Run on a **GPU runtime** and paste back the printed `ITEM 3 SUMMARY` block.

## 1. Setup

In [ ]:
!pip install -q torch-geometric transformers textblob

In [ ]:
import gc
import random
from datetime import datetime
from urllib.parse import urlparse
from typing import Optional

import numpy as np
import pandas as pd
import requests

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset
from torch_geometric.nn import GATConv
from torch_geometric.utils import dense_to_sparse

from transformers import (
    AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
    TextClassificationPipeline, pipeline,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics.pairwise import cosine_similarity
from textblob import TextBlob

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed()

## 2. Data, features, augmentation (replicated from the presented pipeline)

Same fetch → augmentation → feature extraction as the presented
`Fake_news_detection_model_Final_Version.ipynb`: augmentation is applied to the whole
dataset **before** the split (so folds match the defended results), and `corpus_enc`
is kept in the metadata. The temporal columns `year`/`month` are what this ablation
manipulates.

In [ ]:
ROUTE = "https://fake-news-data-extraction.onrender.com/dataset"
data = requests.get(ROUTE).json()
df = pd.DataFrame(data["Scrapped news"])
df['label'] = df['VERACIDAD'].map({'true': 0, 'false': 1, "satira": 2})
df.drop(["METADATA", "VERACIDAD"], axis=1, inplace=True)
print(f"Loaded {len(df)} articles")

In [ ]:
# Clickbait detector + scalar text features (same as the refactored notebook)
cb_tok = AutoTokenizer.from_pretrained("taniwasl/clickbait_es")
cb_mod = AutoModelForSequenceClassification.from_pretrained("taniwasl/clickbait_es")
cb_pipe = TextClassificationPipeline(model=cb_mod, tokenizer=cb_tok, top_k=None)

def sentiment(t): return TextBlob(str(t)).sentiment.polarity
def type_token_ratio(t):
    toks = str(t).split(); return len(set(toks)) / len(toks) if toks else 0.0
def caps_ratio(t):
    t = str(t); return sum(c.isupper() for c in t) / max(len(t), 1)
def excl_ratio(t):
    t = str(t); return t.count('!') / max(len(t), 1)
def clickbait_score(t):
    res = cb_pipe(str(t)[:256])[0]
    return [x for x in res if "click" in x['label'].lower()][0]['score']

META_COLS = ['domain_enc', 'autor_enc', 'corpus_enc', 'year', 'month', 'longitud',
             'polarity', 'clickbait', 'title_len', 'title_excl', 'excl_ratio',
             'caps_ratio', 'ttr']

def extract_features(df):
    df = df.copy()
    df['AUTOR'] = df.get('AUTOR', 'desconocido'); df['AUTOR'] = df['AUTOR'].fillna('desconocido')
    df['URL'] = df.get('URL', 'https://unknown.com'); df['URL'] = df['URL'].fillna('https://unknown.com')
    df['longitud'] = df['CORPUS'].str.split().str.len().astype(float)
    df['polarity'] = df['CORPUS'].apply(sentiment)
    df['ttr'] = df['CORPUS'].apply(type_token_ratio)
    df['caps_ratio'] = df['CORPUS'].apply(caps_ratio)
    df['excl_ratio'] = df['CORPUS'].apply(excl_ratio)
    df['clickbait'] = df['TITULO'].apply(clickbait_score)
    df['title_len'] = df['TITULO'].str.len()
    df['title_excl'] = df['TITULO'].str.count('!')
    df['domain'] = df['URL'].apply(lambda x: urlparse(str(x)).netloc)
    # utc=True + fill 0: matches the presented pipeline (the source FECHA column has
    # mixed timezones, so utc=True is required to parse it).
    df['FECHA'] = pd.to_datetime(df.get('FECHA'), errors='coerce', utc=True)
    df['year'] = df['FECHA'].dt.year.fillna(0).astype(int)
    df['month'] = df['FECHA'].dt.month.fillna(0).astype(int)
    return df

In [ ]:
# Pre-split augmentation, verbatim from the presented pipeline: augment ~30% of the
# non-satire rows on the WHOLE dataset before any split, then extract features on
# everything. This reproduces the defended folds (so the `current` variant matches
# the presented ROCCO numbers).
fill_mask = pipeline("fill-mask", model="dccuchile/bert-base-spanish-wwm-cased", top_k=3)

def augment_text(text, mask_prob=0.15):
    words = str(text).split()
    if len(words) < 5: return text
    n = max(1, int(len(words) * mask_prob))
    for i in random.sample(range(len(words)), n):
        words[i] = "[MASK]"
    masked = " ".join(words)
    try:
        preds = fill_mask(masked)
        if isinstance(preds, list):
            for p in preds:
                masked = masked.replace("[MASK]", p['token_str'], 1)
        return masked
    except Exception:
        return text

subset = df[df["label"] != 2].sample(frac=0.3, random_state=SEED)
aug_df = subset.copy()
aug_df["CORPUS"] = aug_df["CORPUS"].apply(augment_text)
df = pd.concat([df, aug_df], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f"Augmentation complete: added {len(aug_df)} samples (total {len(df)})")

df = extract_features(df)
print("features ready")

## 3. Model + shared helpers

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")
BETO_NAME = "dccuchile/bert-base-spanish-wwm-cased"

class HybridGAT(nn.Module):
    def __init__(self, meta_dim=0):
        super().__init__()
        self.gat1 = GATConv(768, 64, heads=4, concat=True)
        self.gat2 = GATConv(64 * 4, 64, heads=1, concat=False)
        self.cls = nn.Linear(64, 3)
        if meta_dim > 0:
            self.meta_fc = nn.Linear(meta_dim, 32)
            self.cls = nn.Linear(64 + 32, 3)
        self.meta_dim = meta_dim
    def forward(self, meta, text, edge_index):
        x = F.relu(self.gat1(text, edge_index))
        x = F.relu(self.gat2(x, edge_index))
        if self.meta_dim > 0 and meta is not None:
            x = torch.cat([x, F.relu(self.meta_fc(meta))], dim=1)
        return self.cls(x)

def finetune_beto(df_tr, epochs=3, freeze_lower6=True):
    enc = tokenizer(list(df_tr['CORPUS']), truncation=True, padding=True,
                    max_length=256, return_tensors="pt").to(device)
    ytr = torch.tensor(df_tr['label'].values, dtype=torch.long).to(device)
    beto = AutoModel.from_pretrained(BETO_NAME).to(device)
    if freeze_lower6:
        for name, p in beto.named_parameters():
            if any(f"encoder.layer.{i}" in name for i in range(6)):
                p.requires_grad = False
    cls_head = nn.Linear(768, 3).to(device)
    opt = AdamW(list(beto.parameters()) + list(cls_head.parameters()), lr=2e-5)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    loader = DataLoader(TensorDataset(enc['input_ids'], enc['attention_mask'], ytr),
                        batch_size=8, shuffle=True)
    amp = torch.amp.GradScaler('cuda')
    beto.train(); cls_head.train()
    for _ in range(epochs):
        for ids, attn, lab in loader:
            opt.zero_grad()
            with torch.amp.autocast('cuda'):
                out = beto(input_ids=ids, attention_mask=attn)
                loss = crit(cls_head(out.last_hidden_state[:, 0, :]), lab)
            amp.scale(loss).backward(); amp.step(opt); amp.update()
    return beto, cls_head

def get_cls_embeddings(texts, beto, batch=8):
    enc = tokenizer(list(texts), truncation=True, padding=True,
                    max_length=256, return_tensors='pt')
    dl = DataLoader(TensorDataset(enc['input_ids'], enc['attention_mask']), batch_size=batch)
    beto.eval(); out = []
    with torch.no_grad():
        for ix, m in dl:
            h = beto(input_ids=ix.to(device), attention_mask=m.to(device))
            out.append(h.last_hidden_state[:, 0, :].cpu().numpy())
    return np.vstack(out)

## 4. Temporal-view ablation (committee item 3)

The four graph variants on the same 8-fold split, plus two diagnostics:
**(a)** edge-set diff vs the 2-view graph — does the temporal view change *any* edges?
**(b)** edge-composition — are the retained k-NN edges actually temporally close?

In [ ]:
VARIANTS = ["current", "2view", "corrected", "permuted"]
K = 8

set_seed()
skf = StratifiedKFold(n_splits=8, shuffle=True, random_state=SEED)
y = df['label'].values
rng = np.random.RandomState(SEED)

results = {v: {'acc': [], 'f1': []} for v in VARIANTS}
view_pred_frac = {'current': [], 'corrected': []}
edge_close_frac = {'current': [], 'corrected': []}
edge_diff = {v: {'changed': [], 'jaccard': []} for v in VARIANTS if v != '2view'}

for fold, (train_idx, test_idx) in enumerate(skf.split(df, y)):
    print(f"\n=== Fold {fold + 1} ===")
    df_tr = df.iloc[train_idx].copy()
    df_te = df.iloc[test_idx].copy()

    # --- CV-safe encoders (fit on train; unseen -> -1), exactly as presented ---
    le_domain = LabelEncoder().fit(df_tr['domain'])
    le_autor = LabelEncoder().fit(df_tr['AUTOR'])
    le_corpus = LabelEncoder().fit(df_tr['CORPUS'])
    df_tr['domain_enc'] = le_domain.transform(df_tr['domain'])
    df_tr['autor_enc'] = le_autor.transform(df_tr['AUTOR'])
    df_tr['corpus_enc'] = le_corpus.transform(df_tr['CORPUS'])
    df_te['domain_enc'] = df_te['domain'].map(lambda x: le_domain.transform([x])[0] if x in le_domain.classes_ else -1)
    df_te['autor_enc'] = df_te['AUTOR'].map(lambda x: le_autor.transform([x])[0] if x in le_autor.classes_ else -1)
    df_te['corpus_enc'] = df_te['CORPUS'].map(lambda x: le_corpus.transform([x])[0] if x in le_corpus.classes_ else -1)
    mscaler = MinMaxScaler().fit(df_tr[META_COLS])
    meta_all_np = np.vstack([mscaler.transform(df_tr[META_COLS]),
                             mscaler.transform(df_te[META_COLS])])

    # --- real (unscaled) month index, same node order (train then test) ---
    month_idx = np.concatenate([
        (df_tr['year'].values * 12 + df_tr['month'].values).astype(np.float64),
        (df_te['year'].values * 12 + df_te['month'].values).astype(np.float64),
    ])

    # --- fine-tune BETO once per fold, embed both splits ---
    beto, cls_head = finetune_beto(df_tr, epochs=3, freeze_lower6=True)
    Xtr_txt = get_cls_embeddings(df_tr['CORPUS'], beto)
    Xte_txt = get_cls_embeddings(df_te['CORPUS'], beto)
    X_all_txt = np.vstack([Xtr_txt, Xte_txt])

    # --- shared views + the four temporal definitions ---
    sim_text = cosine_similarity(X_all_txt) ** 2
    sim_domain = np.equal.outer(meta_all_np[:, 0], meta_all_np[:, 0]).astype(np.float32)

    # index of 'year' inside META_COLS (scaled) for the degenerate 'current' view
    yr = META_COLS.index('year')
    st_current = (np.abs(np.subtract.outer(meta_all_np[:, yr], meta_all_np[:, yr])) <= 1).astype(np.float32)
    st_corrected = (np.abs(np.subtract.outer(month_idx, month_idx)) <= 1).astype(np.float32)
    month_perm = month_idx[rng.permutation(len(month_idx))]
    st_permuted = (np.abs(np.subtract.outer(month_perm, month_perm)) <= 1).astype(np.float32)
    variant_sim_time = {"current": st_current, "2view": None,
                        "corrected": st_corrected, "permuted": st_permuted}

    n = len(month_idx)
    offdiag = ~np.eye(n, dtype=bool)
    view_pred_frac['current'].append(st_current[offdiag].mean())
    view_pred_frac['corrected'].append(st_corrected[offdiag].mean())
    real_close = st_corrected

    # --- tensors shared across variants ---
    meta_all = torch.tensor(meta_all_np, dtype=torch.float32).to(device)
    t_all = torch.tensor(X_all_txt, dtype=torch.float32).to(device)
    y_all = torch.tensor(np.concatenate([df_tr['label'], df_te['label']]), dtype=torch.long).to(device)
    num_train = len(df_tr)
    train_nodes = torch.arange(num_train, device=device)
    test_nodes = torch.arange(num_train, num_train + len(df_te), device=device)
    true = y_all[test_nodes].cpu().numpy()

    fold_edge_sets = {}
    for v in VARIANTS:
        st = variant_sim_time[v]
        sim = (sim_text + sim_domain) / 2.0 if st is None else (sim_text + sim_domain + st) / 3.0
        np.fill_diagonal(sim, 0)
        sim = np.maximum(sim, sim.T)
        drop = np.argsort(sim, axis=1)[:, :-K]
        for i, rows in enumerate(drop):
            sim[i, rows] = 0
        edge_index, _ = dense_to_sparse(torch.tensor(sim, dtype=torch.float32))

        fold_edge_sets[v] = set(map(tuple, edge_index.t().tolist()))
        if v in ('current', 'corrected'):
            src, dst = edge_index[0].numpy(), edge_index[1].numpy()
            edge_close_frac[v].append(real_close[src, dst].mean())
        edge_index = edge_index.to(device)

        torch.manual_seed(1000 + fold)   # identical GAT init across variants within a fold
        model = HybridGAT(meta_dim=meta_all.shape[1]).to(device)
        gnn_opt = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-3)
        crit2 = nn.CrossEntropyLoss(label_smoothing=0.1)
        for epoch in range(12):
            model.train(); gnn_opt.zero_grad()
            loss2 = crit2(model(meta_all, t_all, edge_index)[train_nodes], y_all[train_nodes])
            loss2.backward(); gnn_opt.step()
        model.eval()
        with torch.no_grad():
            pred = model(meta_all, t_all, edge_index)[test_nodes].argmax(dim=1).cpu().numpy()
        acc = accuracy_score(true, pred); f1 = f1_score(true, pred, average='macro')
        results[v]['acc'].append(acc); results[v]['f1'].append(f1)
        print(f"  {v:<10} Acc={acc:.3f}  F1={f1:.3f}")
        del model, edge_index; torch.cuda.empty_cache()

    # --- edge-set diff vs the 2-view (no-time) graph ---
    base = fold_edge_sets['2view']
    for v in edge_diff:
        this = fold_edge_sets[v]
        edge_diff[v]['changed'].append(len(base ^ this))
        edge_diff[v]['jaccard'].append(len(base & this) / len(base | this))
        print(f"    edges changed vs 2view [{v:<9}] = {len(base ^ this):>6}   "
              f"Jaccard={len(base & this) / len(base | this):.4f}")

    del beto, cls_head; torch.cuda.empty_cache(); gc.collect()

## 5. Summary

In [ ]:
print("\n================ ITEM 3 SUMMARY ================")
print(f"{'Graph variant':<26}{'Acc':>14}{'Macro-F1':>16}")
for v in VARIANTS:
    a = np.array(results[v]['acc']); f = np.array(results[v]['f1'])
    print(f"{v:<26}{a.mean():.3f}+/-{a.std():.3f}   {f.mean():.3f}+/-{f.std():.3f}")

print("\n---- Edge-set diff vs the 2-view (no-time) graph ----")
print("If a temporal view changes 0 edges (Jaccard=1.0000) it CANNOT affect the model.")
for v in edge_diff:
    ch = np.array(edge_diff[v]['changed']); jc = np.array(edge_diff[v]['jaccard'])
    print(f"  {v:<10} changed/fold = {ch.mean():.1f}   Jaccard = {jc.mean():.4f}")

print("\n---- Edge-composition diagnostic ----")
print("Off-diagonal pairs flagged temporally-close BY THE VIEW's own predicate:")
print(f"  current  (scaled year): {np.mean(view_pred_frac['current']):.4f}   <- expect ~1.0000 (degenerate)")
print(f"  corrected (real month): {np.mean(view_pred_frac['corrected']):.4f}")
print("Retained k-NN edges that are temporally close (real |d_month| <= 1):")
print(f"  current graph:   {np.mean(edge_close_frac['current']):.4f}")
print(f"  corrected graph: {np.mean(edge_close_frac['corrected']):.4f}")